## 3 Whisper Finetuning
 
This notebook finetunes Whisper on all Romansh idioms with all the training data.

In [ ]:
import os
import sys
from pathlib import Path
import torch
from transformers import Seq2SeqTrainer
from functools import partial

# Ensure the package is importable
notebook_dir = Path.cwd()
whisper_dir = notebook_dir.parent
sys.path.append(str(whisper_dir))

from whisper_asr import (
    load_idiom_data,
    OnTheFlyDataset,
    collate_fn,
    load_model_and_processor,
    compute_metrics,
    get_training_args,
)
from whisper_asr.constants import MODELS_ROOT

# Disable tokenizer parallelism warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device("cuda")
print(f"Using device: {DEVICE}")

First the train and validation data is loaded.

In [ ]:
print("Loading all Romansh idiom datasets...")
train_samples, val_samples = load_idiom_data()
print(f"Train samples: {len(train_samples)}, Validation samples: {len(val_samples)}")

Then the Whisper components are loaded.

In [ ]:
MODEL_NAME = "openai/whisper-medium"
OUTPUT_DIR = MODELS_ROOT / "whisper-medium-rm"

model, feature_extractor, tokenizer, processor = load_model_and_processor(
    MODEL_NAME, DEVICE, language="it"
)

print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

Datasets are prepared to process audio on the fly during training to reduce memory usage.

In [ ]:
train_dataset = OnTheFlyDataset(
    train_samples,          # list of dicts
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    language="it",
    task="transcribe",
)
eval_dataset = OnTheFlyDataset(
    val_samples,
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    language="it",
    task="transcribe",
)
print(f"Train dataset size: {len(train_dataset)}, Eval dataset size: {len(eval_dataset)}")

We get the training arguments and the compute metrics.

In [ ]:
training_args = get_training_args(OUTPUT_DIR)
print("Training arguments:")
print(training_args)
compute_metrics_fn = partial(compute_metrics, tokenizer=tokenizer)

Then we initialize the trainer.

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics_fn,
)
print("Trainer initialized.")

Finally we can start training.

In [ ]:
print("Starting training...")
trainer.train()

Finally we save the model.

In [ ]:
print(f"Saving model to {OUTPUT_DIR}")
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Done.")